### the intention here is to build the workflow where we gather data response from the llm about whats in the pdf guia, that was proven before that it can be accurate, in notebook 04, and then compare with the field DS_PROCEDIMENTO and see if they match or not, generating another df with a aut_validacao_status. we can t keep ds contato , so in the workflow we check if theres a analysis form the analist or not for comparing only purposes of the past data

## Part 0: getting the right avisos from enriched excel file.

In [22]:
# download file from s3 bucket = ,S3_BUCKET_NAME="agente-ai-laudos" , folder = segunda_saida/, file_name = "consolidado_enriquecido.xlsx"
# excel_handler = S3ExcelClient(homolog=False)


downlaod excel file segunda_saida e terceira saida

In [23]:
# ...

we should add code to , in gathering the avisos of interest to process,  uses the  ones of terceira saida a well? like
first take the avisos of interest from segunda_saida, they all were not finalized yet gareanteed. them we compare with the ones from terceira_saida, to gather only the avisos that we not have processed yet there. so our list of avisso its just whats needed. in the end, we update the terceira_saida excel file.

extract avisos from eexcel

In [1]:
import pandas as pd
from typing import List, Tuple, Optional

def extract_avisos_from_tabs(
    xls: pd.ExcelFile, 
    tab_names: List[str], 
    mask_criteria: dict, 
    limit: Optional[int] = None
) -> Tuple[List[int], str]:
    """
    Extract avisos from multiple Excel tabs based on mask criteria and limit.

    Args:
        xls (pd.ExcelFile): Excel file object
        tab_names (List[str]): List of Excel sheet/tab names
        mask_criteria (dict): Dictionary with column name and values for filtering
        limit (int, optional): Maximum number of avisos to extract (takes last N records per tab)

    Returns:
        Tuple[List[int], str]: Tuple containing:
            - Combined list of aviso IDs (CD_AVISO_CIRURGIA) from all tabs
            - String representation of avisos for SQL queries
    """
    combined_avisos = []

    for tab_name in tab_names:
        try:
            df = pd.read_excel(xls, sheet_name=tab_name)
        except Exception as e:
            print(f"❌ Error reading tab '{tab_name}': {e}")
            continue

        mask = pd.Series([True] * len(df))
        for column, values in mask_criteria.items():
            if column in df.columns:
                if callable(values):
                    mask = mask & df[column].apply(values)
                elif isinstance(values, set):
                    mask = mask & df[column].isin(values)
                else:
                    mask = mask & (df[column] == values)

        filtered_avisos = df.loc[mask, "CD_AVISO_CIRURGIA"]

        if limit and len(filtered_avisos) > limit:
            filtered_avisos = filtered_avisos.tail(limit)

        avisos_list = filtered_avisos.tolist()
        print(f"📋 Found {len(avisos_list)} records to process for {tab_name}")
        if len(avisos_list) == 0:
            print(f"   ℹ️  No matching records found in {tab_name}")

        combined_avisos.extend(avisos_list)

    avisos_str = ", ".join(str(i) for i in combined_avisos)
    return combined_avisos, avisos_str

In [2]:
import pandas as pd
excel_path = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/excel/consolidado_enriquecido (4).xlsx"

xls = pd.ExcelFile(excel_path)
valid_friendly = {"Contorno", "Betim-Contagem", "Salvador/Bahia", "Nova Lima"}
mask_criteria = {"friendly_name": valid_friendly, "DT_FINALIZADO_MMD": "NÃO FINALIZADO"}

combined_avisos, avisos_str = extract_avisos_from_tabs(
    xls=xls,
    tab_names=["Autorizados", "Pendentes", "Solicitados"],
    mask_criteria=mask_criteria,
    limit=None
)

📋 Found 26 records to process for Autorizados
📋 Found 56 records to process for Pendentes
📋 Found 68 records to process for Solicitados


# datab base querys

In [61]:
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.services.database.oracle import OracleService
from app.services.database.mariadb import MariaDBService
from app.utils.db_operations import load_query_from_file, execute_query_to_df
from app.utils.config import load_config
from app.utils.logger import get_logger


logger = get_logger(name=__name__)
config_vars = load_config()

mount the query string

In [67]:
# load the query strings
autorizacao_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_autorizacao.sql"
autorizacao_query_str = load_query_from_file(autorizacao_query_file)
autorizacao_sql_final = autorizacao_query_str.replace("?", avisos_str, 1)

procedimento_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_procedimento.sql"
procedimento_query_str = load_query_from_file(procedimento_query_file)
procedimento_sql_final = procedimento_query_str.replace("?", avisos_str, 1)

finalizado_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_pedido_finalizado_resposta.sql"
finalizado_query_str = load_query_from_file(finalizado_query_file)
finalizado_sql_final = finalizado_query_str.replace("?", avisos_str, 1)

aviso_cirurgia_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_aviso_cirurgia.sql"
aviso_cirurgia_query_str = load_query_from_file(aviso_cirurgia_query_file)
aviso_cirurgia_sql_final = aviso_cirurgia_query_str.replace("?", avisos_str, 1)

In [68]:
autorizacao_sql_final

'SELECT guia.cd_guia,\n       guia.tp_guia,\n       guia.cd_aviso_cirurgia,\n       guia.tp_situacao,\n       guia.dt_solicitacao,\n       guia.dt_autorizacao,\n       overmind_aux_log_pre.cd_guia,\n       overmind_aux_log_pre.cd_aviso,\n       overmind_aux_log_pre.dt_insercao,\n       overmind_aux_log_pre.dt_overmind,\n       overmind_aux_log_pre.ds_protocolo,\n       overmind_aux_log_pre.dt_status,\n       overmind_aux_log_pre.tp_status,\n       overmind_aux_log_pre.ds_status,\n       overmind_aux_log_pre.cd_senha,\n       overmind_aux_log_pre.dt_senha,\n       overmind_aux_log_pre.ds_guia_path   \n       FROM dbamv.guia\n       LEFT JOIN dbahmd.overmind_aux_log_pre ON overmind_aux_log_pre.cd_guia = guia.cd_guia\n          WHERE guia.cd_aviso_cirurgia IN (852029, 857416, 857469, 857749, 857883, 858315, 858565, 859112, 859542, 859818, 859989, 860274, 860428, 860565, 861063, 861067, 861468, 861565, 862058, 862378, 862596, 862762, 862922, 862934, 863419, 863440, 863507, 863624, 863647, 

In [64]:
maria_db_procedimento_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=procedimento_sql_final)
oracle_db_autorizacao_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=autorizacao_sql_final)
oracle_db_finalizado_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=finalizado_sql_final)
maria_db_aviso_cirurgia_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=aviso_cirurgia_sql_final)

{"timestamp": "2025-09-01T15:43:48", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection established to\n                srvawsdb002.cow7tj30bxpl.us-east-1.rds.amazonaws.com:3306/", "filename": "mariadb.py", "lineno": 27}
{"timestamp": "2025-09-01T15:43:48", "level": "INFO", "name": "app.services.database.mariadb", "message": "Executing MariaDB query...", "filename": "mariadb.py", "lineno": 67}
{"timestamp": "2025-09-01T15:43:48", "level": "INFO", "name": "app.services.database.mariadb", "message": "Executing MariaDB query...", "filename": "mariadb.py", "lineno": 67}
{"timestamp": "2025-09-01T15:43:48", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ Fetched 126 rows from MariaDB.", "filename": "mariadb.py", "lineno": 75}
{"timestamp": "2025-09-01T15:43:48", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection closed.", "filename": "mariadb.py", "lineno": 41}
{"timestamp": "2025-09-01T1

## getting a single table with the info we need

In [31]:
import pandas as pd

maria_db_procedimento_df_processed = maria_db_procedimento_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA", "procedure": "DS_PROCEDIMENTO"})
maria_db_procedimento_df_processed.drop(columns=["hospitalization_type"], inplace=True)
maria_db_procedimento_df_processed = maria_db_procedimento_df_processed[maria_db_procedimento_df_processed["DS_PROCEDIMENTO"].notna()]
maria_db_procedimento_df_processed.reset_index(drop=True, inplace=True)

autorizacao_no_need_cols = [col for col in oracle_db_autorizacao_df.columns if col not in ["CD_AVISO_CIRURGIA", "CD_GUIA", "CD_SENHA", "DS_GUIA_PATH", "DS_PROTOCOLO"]]
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df.drop(columns=autorizacao_no_need_cols)
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df_processed[oracle_db_autorizacao_df_processed["DS_GUIA_PATH"].notna()]
oracle_db_autorizacao_df_processed.reset_index(drop=True, inplace=True)

finalizado_no_need_cols = [col for col in oracle_db_finalizado_df.columns if col not in ["CD_REGISTRO_VINCULADO", "DS_CONTATO"]]
oracle_db_finalizado_df_processed = oracle_db_finalizado_df.drop(columns=finalizado_no_need_cols)
oracle_db_finalizado_df_processed.rename(columns={"CD_REGISTRO_VINCULADO": "CD_AVISO_CIRURGIA"}, inplace=True)
oracle_db_finalizado_df_processed = oracle_db_finalizado_df_processed[oracle_db_finalizado_df_processed["DS_CONTATO"].notna()]
oracle_db_finalizado_df_processed.reset_index(drop=True, inplace=True)

maria_db_aviso_cirurgia_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA"}, inplace=True)
keep_cols = ["CD_AVISO_CIRURGIA", "health_insurance_name"]
maria_db_aviso_cirurgia_df = maria_db_aviso_cirurgia_df[keep_cols]
maria_db_aviso_cirurgia_df.reset_index(drop=True, inplace=True)

In [32]:
merged_df_autorizacao = pd.merge(
    maria_db_procedimento_df_processed,
    oracle_db_autorizacao_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

merged_df_autorizacao_final = pd.merge(
    merged_df_autorizacao,
    oracle_db_finalizado_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="left"
)

final_extracted_df = pd.merge(
    merged_df_autorizacao_final,
    maria_db_aviso_cirurgia_df,
    on="CD_AVISO_CIRURGIA",
    how="left"
)

In [ ]:
final_extracted_df.head()

In [34]:
len(merged_df_autorizacao)

54

In [35]:
# entering the pdf link, downloading the pdf, adding as another col
import requests
import numpy as np
def fetch_pdf_bytes(url):
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200 and 'application/pdf' in response.headers.get('content-type', ''):
            return response.content
        else:
            return np.nan
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return np.nan

# Apply to all links in DS_GUIA_PATH
final_extracted_df['DS_PDF_BYTES'] = final_extracted_df['DS_GUIA_PATH'].apply(fetch_pdf_bytes)
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,DS_PROTOCOLO,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES
0,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,121223221,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...
1,857883,"[{""code"":30205050,""description"":""AMIGDALECTOMI...",19506292.0,121223228,J5VEW29,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...
2,859112,"[{""code"":30501350,""description"":""RINOSSEPTOPLA...",19537543.0,203941226,5045663575,https://cdns.overmind.ai/autorizacao-sulameric...,\n Procedimento Autorizado\n Pacient...,SUL AMERICA,b'%PDF-1.4\n%\xf6\xe4\xfc\xdf\n1 0 obj\n<<\n/T...


In [36]:
# Create a new column with clickable links for DS_GUIA_PATH
def make_clickable(url):
    if pd.notna(url):
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""
final_extracted_df['DS_GUIA_PATH_CLICKABLE'] = final_extracted_df['DS_GUIA_PATH'].apply(make_clickable)
from IPython.display import display, HTML
display(HTML(final_extracted_df[['DS_GUIA_PATH', 'DS_GUIA_PATH_CLICKABLE']].head(10).to_html(escape=False)))

,DS_GUIA_PATH,DS_GUIA_PATH_CLICKABLE
0,https://cdns.overmind.ai/autorizacao-bradesco-121223221-1752237144964.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223221-1752237144964.pdf
1,https://cdns.overmind.ai/autorizacao-bradesco-121223228-1753454043822.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223228-1753454043822.pdf
2,https://cdns.overmind.ai/autorizacao-sulamerica-203941226-1753068369928.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-203941226-1753068369928.pdf
3,https://cdns.overmind.ai/autorizacao-bradesco-121612505-1753983787863.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121612505-1753983787863.pdf
4,https://cdns.overmind.ai/autorizacao-sulamerica-204937675-1754852532865.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-204937675-1754852532865.pdf
5,https://cdns.overmind.ai/autorizacao-sulamerica-205226366-1755799999288.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-205226366-1755799999288.pdf
6,https://cdns.overmind.ai/autorizacao-sulamerica-205226406-1754242864640.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-205226406-1754242864640.pdf
7,https://cdns.overmind.ai/autorizacao-bradesco-121847668-1753971062828.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121847668-1753971062828.pdf
8,https://cdns.overmind.ai/autorizacao-bradesco-121937018-1753877009759.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121937018-1753877009759.pdf
9,https://cdns.overmind.ai/autorizacao-sulamerica-205429952-1756134195081.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-205429952-1756134195081.pdf


# part 2: inputing data to technique, observing the outputed data

In [37]:
# trying to use the mudular code to get the workflow feeling
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.utils.process_images import process_blobs
from app.utils.textract_service import TextractPDFAnalyzer
from app.utils.llm_service import AnthropicLLMService
from app.utils.aws_services_handler import create_boto3_client
from app.utils.config import load_config, AppConstants
from app.utils.logger import get_logger
from app.utils.system_prompts.autorizacao_prompt import Prompts


logger = get_logger(name=__name__)
config_vars = load_config()

In [38]:
app_constants = AppConstants()
textract_client = create_boto3_client("textract", config_vars)
bedrock_client = create_boto3_client("bedrock-runtime", config_vars)
s3_client = create_boto3_client("s3", config_vars)
bucket_name = "autorizacoes"

# Initialize the TextractPDFAnalyzer
pdf_analyzer = TextractPDFAnalyzer(
    textract_client=textract_client,
    s3_client=s3_client,
    bucket_name="autorizacoes-textract",
    bucket_folder="guias-pdf"
)

autorizacao_prompt = Prompts.autorizacao_extraction_prompt


llm_instance = AnthropicLLMService(
    model_id=app_constants.BEDROCK_DEFAULT_MODEL_ID,
    model_version=app_constants.BEDROCK_DEFAULT_MODEL_VERSION,
    client=bedrock_client,
    system_prompt=autorizacao_prompt,
    max_tokens=app_constants.MAX_TOKENS,
    temperature=app_constants.TEMPERATURE,
    budget_tokens=app_constants.BUDGET_TOKENS
)

{"timestamp": "2025-09-01T11:26:45", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-09-01T11:26:45", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-09-01T11:26:45", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-09-01T11:26:45", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-09-01T11:26:45", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-eas

In [39]:
# llm processing

from typing import Dict, Tuple
# Process rows from the dataframe 
def llm_extraction(df: pd.DataFrame, limit=None) -> Tuple[Dict, Dict]:
    """
    Extract data from PDFs using Textract and process with LLM.
    
    Args:
        df (pd.DataFrame): DataFrame containing PDF information
        limit (int, optional): Limit number of rows to process
        
    Returns:
        Tuple[Dict, Dict]: LLM results and Textract results dictionaries
    """
    if limit:
        df = df[:limit]
    textract_results = {}
    llm_results = {}
    for index, row in df.iterrows():
        pdf_bytes = row['DS_PDF_BYTES']
        file_id = str(row['CD_AVISO_CIRURGIA'])
        
        if pd.isna(pdf_bytes):
            print(f"Skipping row {index}: No PDF bytes available")
            continue
        
        # Extract forms data using Textract
        forms_data = pdf_analyzer.extract_forms_data_from_pdf(
            pdf_bytes=pdf_bytes,
            file_id=file_id
        )
        textract_results[file_id] = forms_data
        
        
        # Process with LLM
        forms_data_str = str(forms_data)
        llm_response = llm_instance.invoke_model(input_str=forms_data_str)
        llm_results[file_id] = llm_response
    return llm_results, textract_results

def llm_dict_to_final_df(llm_results: Dict, textract_results: Dict, df: pd.DataFrame, limit=None) -> pd.DataFrame:
    """
    Convert LLM and Textract results dictionaries to final merged DataFrame.
    
    Args:
        llm_results (Dict): Dictionary with LLM processing results
        textract_results (Dict): Dictionary with Textract extraction results
        df (pd.DataFrame): Original DataFrame to merge with
        limit (int, optional): Limit number of rows from original DataFrame
        
    Returns:
        pd.DataFrame: Merged DataFrame with LLM and Textract results
    """
    if limit:
        df = df[:limit]

    llm_results_df = pd.DataFrame.from_dict(llm_results, orient="index")
    llm_results_df.index.name = "CD_AVISO_CIRURGIA"
    llm_results_df = llm_results_df.reset_index()

    # Expand LLM results into separate columns
    llm_expanded_df = pd.json_normalize(llm_results_df.iloc[:, 1:].to_dict('records'))
    llm_expanded_df['CD_AVISO_CIRURGIA'] = llm_results_df['CD_AVISO_CIRURGIA']

    # Add suffix to distinguish LLM columns
    llm_expanded_df = llm_expanded_df.add_suffix('_llm').rename(columns={'CD_AVISO_CIRURGIA_llm': 'CD_AVISO_CIRURGIA'})


    # add the full llm results as a new column, maping the llm dict to the CD_AVISO_CIRURGIA
    llm_expanded_df['llm_full_results'] = llm_expanded_df['CD_AVISO_CIRURGIA'].map(llm_results)
    # Convert CD_AVISO_CIRURGIA to int to match the original dataframe type
    llm_expanded_df['CD_AVISO_CIRURGIA'] = llm_expanded_df['CD_AVISO_CIRURGIA'].astype(int)
    # Map textract results based on CD_AVISO_CIRURGIA values
    # Convert the keys in textract_results to int for proper mapping
    textract_results_int_keys = {int(k): v for k, v in textract_results.items()}
    llm_expanded_df['textract_results'] = llm_expanded_df['CD_AVISO_CIRURGIA'].map(textract_results_int_keys)

    # Ensure original df has the correct data type
    df = df.copy()
    df['CD_AVISO_CIRURGIA'] = df['CD_AVISO_CIRURGIA'].astype(int)

    # Merge df with llm results
    merged_df = df.merge(
        llm_expanded_df,
        on='CD_AVISO_CIRURGIA',
        how='inner'
    )
    # Inner join with original dataframe
    return merged_df

In [ ]:
llm_results, textract_results = llm_extraction(df=final_extracted_df)
llm_results_df_run = llm_dict_to_final_df(llm_results=llm_results, textract_results=textract_results, df=final_extracted_df)

print(f"Processing completed! Created textract_eval_df with {len(llm_results_df_run)} rows")
print(f"LLM columns added: {[col for col in llm_results_df_run.columns if col.endswith('_llm')]}")
llm_results_df_run.head()



# test from here

In [41]:
llm_results_df_run.loc[0, 'llm_full_results']

{'procedimentos_autorizados': ['VARIZES TRATAMENTO CIRURGICO DE DOIS MEMBROS'],
 'paciente': 'BARBARA ARAUJO SCHNNEPPEL CORREA',
 'codigo_autorizado': '30907136 x 1',
 'senha': 'J5VEYT7',
 'validade_senha': None,
 'data_solicitacao': '11/07/2025',
 'observacoes_opme': 'Pedido sem OPME',
 'observacoes_gerais': None,
 'profissional_solicitante': 'MATEUS ALVES BORGES CRISTINO'}

In [42]:
llm_results_df = llm_results_df_run.copy()
llm_results_df.loc[0, 'CD_AVISO_CIRURGIA']

np.int64(857469)

In [ ]:
# Create a new column with clickable links for DS_GUIA_PATH
def make_clickable(url):
    if pd.notna(url):
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""
llm_results_df['DS_GUIA_PATH_CLICKABLE'] = llm_results_df['DS_GUIA_PATH'].apply(make_clickable)
from IPython.display import display, HTML
display(HTML(llm_results_df[['CD_AVISO_CIRURGIA','DS_GUIA_PATH', 'DS_GUIA_PATH_CLICKABLE']].head(10).to_html(escape=False)))

# Part 3: comparing the data from the llm with the field DS_PROCEDIMENTO, generating a new field validacao_status with the values "match" or "not match"

The new fields cols would be  validacao_status_codigo_procedimento, validacao_status_descricao_procedimento


In [44]:
# Part 3: comparing the data from the llm with the field DS_PROCEDIMENTO, generating a new field validacao_status with the values "match" or "not match"

# The new fields cols would be  validacao_status_codigo_procedimento, validacao_status_descricao_procedimento
# example of the field DS_PROCEDIMENTO:
# '[{"code":30205050,"description":"AMIGDALECTOMIA DAS PLATINAS","isMain":true,"pro_fat_id":"30205050","procedure_code":30205050,"quantity":1,"surgery_id":1227},{"code":30205271,"description":"ADENOIDECTOMIA POR VIDEOENDOSCOPIA","isMain":false,"pro_fat_id":"30205271","procedure_code":30205271,"quantity":1,"surgery_id":5974},{"code":30205069,"description":"AMIDALECTOMIA LINGUAL","isMain":false,"pro_fat_id":"30205069","procedure_code":30205069,"quantity":1,"surgery_id":1236},{"code":30501458,"description":"TURBINECTOMIA OU TURBINOPLASTIA - UNILATERAL","isMain":false,"pro_fat_id":"30501458","procedure_code":30501458,"quantity":2,"surgery_id":1191}]'
# example of the field 'codigo_autorizado_llm':
# '30205050 x 1, 30205069 x 1, 30205271 x 1, 30501458 x 2'




import pandas as pd
import json

import pandas as pd
import json
from typing import List, Tuple

def validate_codigo_procedimento(
    ds_procedimento_col: List[str], 
    codigo_autorizado_llm_col: List[str]
) -> Tuple[List[str], List[str]]:
    """
    Compares DS_PROCEDIMENTO and codigo_autorizado_llm columns to generate validation status and explanations.

    Args:
        ds_procedimento_col (List[str]): List of DS_PROCEDIMENTO JSON strings.
        codigo_autorizado_llm_col (List[str]): List of LLM output strings (e.g., '30205050 x 1, 30205069 x 1').

    Returns:
        Tuple[List[str], List[str]]: Tuple containing:
            - List of validation statuses ("match" or "not match") for each row
            - List of explanations describing what doesn't match
    """
    def normalize_quantity(qty_str):
        """Normalize quantity strings to handle '1' vs '01' cases"""
        return str(int(qty_str)).strip()
    
    def parse_ds_procedimento(json_str):
        try:
            data_list = json.loads(json_str)
            return set(
                (str(item.get('procedure_code', item.get('code'))), normalize_quantity(str(item['quantity'])))
                for item in data_list
            )
        except (json.JSONDecodeError, TypeError):
            return set()

    def parse_llm_data(llm_str):
        if pd.isna(llm_str):
            return set()
        try:
            items = llm_str.split(', ')
            parsed_items = []
            for item in items:
                parts = item.split(' x ')
                if len(parts) == 2:
                    code = parts[0].strip()
                    qty = normalize_quantity(parts[1].strip())
                    parsed_items.append((code, qty))
            return set(parsed_items)
        except (IndexError, ValueError):
            return set()

    def compare_data_with_explanation(ds_set, llm_set):
        if ds_set.issubset(llm_set):
            return "correspondente", "Todos os procedimentos de DS_PROCEDIMENTO estão presentes na nos campos de guia extraidos pelo agente"
        
        # Find missing procedures
        missing_in_llm = ds_set - llm_set
        extra_in_llm = llm_set - ds_set
        
        explanations = []
        
        if missing_in_llm:
            missing_codes = [f"código {code} x {qty}" for code, qty in missing_in_llm]
            explanations.append(f"Ausente no LLM: {', '.join(missing_codes)}")
        
        if extra_in_llm:
            extra_codes = [f"código {code} x {qty}" for code, qty in extra_in_llm]
            explanations.append(f"Extra no LLM: {', '.join(extra_codes)}")
        
        return "não correspondente", "; ".join(explanations)

    results = [
        compare_data_with_explanation(
            parse_ds_procedimento(ds_proc), 
            parse_llm_data(llm_val)
        )
        for ds_proc, llm_val in zip(ds_procedimento_col, codigo_autorizado_llm_col)
    ]
    
    statuses = [result[0] for result in results]
    explanations = [result[1] for result in results]
    
    return statuses, explanations

In [45]:
# Get both validation status and explanation
validation_status, validation_explanation = validate_codigo_procedimento(
    llm_results_df['DS_PROCEDIMENTO'].tolist(),
    llm_results_df['codigo_autorizado_llm'].tolist()
)

# Add both columns to the dataframe
llm_results_df['validacao_status_codigo_procedimento'] = validation_status
llm_results_df['validacao_explanation_codigo_procedimento'] = validation_explanation



In [46]:
# loc result of second row, of value from col llm_full_results
llm_results_df.iloc[1]['llm_full_results']

{'procedimentos_autorizados': ['AMIGDALECTOMIA DAS PALATINAS',
  'AMIGDALECTOMIA LINGUAL',
  'ADENOIDECTOMIA POR VIDEOENDOSCOPIA',
  'TURBINECTOMIA OU TURBINOPLASTIA UNILATERAL'],
 'paciente': 'MILENA RAFAELA TRINDADE',
 'codigo_autorizado': '30205050 x 1, 30205069 x 1, 30205271 x 1, 30501458 x 2',
 'senha': 'J5VEW29',
 'validade_senha': None,
 'data_solicitacao': '11/07/2025',
 'observacoes_opme': 'Pedido sem OPME',
 'observacoes_gerais': 'Solicitacao de autorizacao',
 'profissional_solicitante': 'AURELIA ALBUQUERQUE MARTINS'}

In [47]:
import pandas as pd
from datetime import timedelta
import numpy as np

def process_dates_by_row(df: pd.DataFrame) -> pd.DataFrame:
    """
    Process date columns row by row to ensure proper formatting and calculate validade_senha_llm.
    This function works with the individual _llm columns before they get filtered out.
    """
    # Make a copy to avoid modifying the original dataframe
    df_copy = df.copy()
    
    # Process each row individually
    for index, row in df_copy.iterrows():
        # Get the individual column values
        data_solicitacao = row.get('data_solicitacao_llm')
        validade_senha = row.get('validade_senha_llm') 
        health_insurance = row.get('health_insurance_name', '')
        
        # Process data_solicitacao_llm
        if pd.notna(data_solicitacao) and data_solicitacao not in [None, 'None', '']:
            try:
                # Try to parse as datetime if it's a string
                if isinstance(data_solicitacao, str):
                    # Handle various date formats
                    for fmt in ['%d/%m/%Y', '%d-%m-%Y', '%Y-%m-%d']:
                        try:
                            parsed_date = pd.to_datetime(data_solicitacao, format=fmt)
                            df_copy.at[index, 'data_solicitacao_llm'] = parsed_date.strftime('%d-%m-%Y')
                            break
                        except:
                            continue
                    else:
                        # If no format worked, try auto-parsing
                        try:
                            parsed_date = pd.to_datetime(data_solicitacao, errors='coerce')
                            if pd.notna(parsed_date):
                                df_copy.at[index, 'data_solicitacao_llm'] = parsed_date.strftime('%d-%m-%Y')
                            else:
                                df_copy.at[index, 'data_solicitacao_llm'] = None
                        except:
                            df_copy.at[index, 'data_solicitacao_llm'] = None
                elif isinstance(data_solicitacao, pd.Timestamp):
                    df_copy.at[index, 'data_solicitacao_llm'] = data_solicitacao.strftime('%d-%m-%Y')
            except:
                df_copy.at[index, 'data_solicitacao_llm'] = None
        else:
            df_copy.at[index, 'data_solicitacao_llm'] = None
            
        # Process validade_senha_llm
        if pd.notna(validade_senha) and validade_senha not in [None, 'None', '']:
            try:
                # Try to parse as datetime if it's a string
                if isinstance(validade_senha, str):
                    # Handle various date formats
                    for fmt in ['%d/%m/%Y', '%d-%m-%Y', '%Y-%m-%d']:
                        try:
                            parsed_date = pd.to_datetime(validade_senha, format=fmt)
                            df_copy.at[index, 'validade_senha_llm'] = parsed_date.strftime('%d-%m-%Y')
                            break
                        except:
                            continue
                    else:
                        # If no format worked, try auto-parsing
                        try:
                            parsed_date = pd.to_datetime(validade_senha, errors='coerce')
                            if pd.notna(parsed_date):
                                df_copy.at[index, 'validade_senha_llm'] = parsed_date.strftime('%d-%m-%Y')
                            else:
                                df_copy.at[index, 'validade_senha_llm'] = None
                        except:
                            df_copy.at[index, 'validade_senha_llm'] = None
                elif isinstance(validade_senha, pd.Timestamp):
                    df_copy.at[index, 'validade_senha_llm'] = validade_senha.strftime('%d-%m-%Y')
            except:
                df_copy.at[index, 'validade_senha_llm'] = None
        else:
            # For BRADESCO, calculate validade_senha from data_solicitacao + 180 days
            if ((health_insurance in ['BRADESCO', 'BRADESCO OPERADORA']) and 
                pd.notna(df_copy.at[index, 'data_solicitacao_llm']) and 
                df_copy.at[index, 'data_solicitacao_llm'] not in [None, 'None', '']):
                try:
                    data_solicitacao_formatted = df_copy.at[index, 'data_solicitacao_llm']
                    if isinstance(data_solicitacao_formatted, str):
                        base_date = pd.to_datetime(data_solicitacao_formatted, format='%d-%m-%Y')
                        validade_date = base_date + timedelta(days=180)
                        df_copy.at[index, 'validade_senha_llm'] = validade_date.strftime('%d-%m-%Y')
                    else:
                        df_copy.at[index, 'validade_senha_llm'] = None
                except:
                    df_copy.at[index, 'validade_senha_llm'] = None
            else:
                df_copy.at[index, 'validade_senha_llm'] = None
    
    return df_copy

In [48]:
llm_results_df = process_dates_by_row(llm_results_df)

In [49]:
llm_results_df.loc[1, 'validade_senha_llm']

'07-01-2026'

In [50]:
print("Before llm_full_results reconstruction:")
print(f"validade_senha_llm: {llm_results_df.loc[0, 'validade_senha_llm']}")
print(f"data_solicitacao_llm: {llm_results_df.loc[0, 'data_solicitacao_llm']}")

Before llm_full_results reconstruction:
validade_senha_llm: 07-01-2026
data_solicitacao_llm: 11-07-2025


In [51]:
llm_results_df["llm_full_results"] = llm_results_df.apply(lambda row: {
    "procedimentos_autorizados": row.get("procedimentos_autorizados_llm", []) or [],
    "paciente": row.get("paciente_llm", None),
    "codigo_autorizado": row.get("codigo_autorizado_llm", None),
    "senha": row.get("senha_llm", None),
    "validade_senha": (
        str(row.get("validade_senha_llm")) if pd.notna(row.get("validade_senha_llm")) else None
    ),
    "data_solicitacao": (
        str(row.get("data_solicitacao_llm")) if pd.notna(row.get("data_solicitacao_llm")) else None
    ),
    "observacoes_opme": row.get("observacoes_opme_llm", None),
    "observacoes_gerais": row.get("observacoes_gerais_llm", None),
    "profissional_solicitante": row.get("profissional_solicitante_llm", None)
}, axis=1)

In [52]:
llm_results_df.loc[0, 'validade_senha_llm']

'07-01-2026'

In [53]:
llm_results_df.loc[0, 'llm_full_results']

{'procedimentos_autorizados': ['VARIZES TRATAMENTO CIRURGICO DE DOIS MEMBROS'],
 'paciente': 'BARBARA ARAUJO SCHNNEPPEL CORREA',
 'codigo_autorizado': '30907136 x 1',
 'senha': 'J5VEYT7',
 'validade_senha': '07-01-2026',
 'data_solicitacao': '11-07-2025',
 'observacoes_opme': 'Pedido sem OPME',
 'observacoes_gerais': None,
 'profissional_solicitante': 'MATEUS ALVES BORGES CRISTINO'}

In [54]:
# final processing response: 
keep_cols = ['CD_AVISO_CIRURGIA', 'DS_GUIA_PATH',  'DS_PROTOCOLO', 'llm_full_results', 'DS_CONTATO', 'DS_PROCEDIMENTO', 'validacao_status_codigo_procedimento', 'validacao_explanation_codigo_procedimento']
llm_results_df = llm_results_df[keep_cols]
#Rename the cols:
llm_results_df.columns = ['CD_AVISO_CIRURGIA', 'DS_GUIA_PATH', 'DS_PROTOCOLO', 'AGENT RESULTS', 'DS_CONTATO', 'DS_PROCEDIMENTO', 'VALIDACAO_CODIGO_PROCEDIMENTO', 'VALIDACAO_EXPLICACAO']
llm_results_df.head(3)

,CD_AVISO_CIRURGIA,DS_GUIA_PATH,DS_PROTOCOLO,AGENT RESULTS,DS_CONTATO,DS_PROCEDIMENTO,VALIDACAO_CODIGO_PROCEDIMENTO,VALIDACAO_EXPLICACAO
0,857469,https://cdns.overmind.ai/autorizacao-bradesco-...,121223221,{'procedimentos_autorizados': ['VARIZES TRATAM...,\n Procedimento Autorizado\n Pacient...,"[{""code"":30907136,""description"":""VARIZES - TRA...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...
1,857883,https://cdns.overmind.ai/autorizacao-bradesco-...,121223228,{'procedimentos_autorizados': ['AMIGDALECTOMIA...,\n Procedimento Autorizado\n Pacient...,"[{""code"":30205050,""description"":""AMIGDALECTOMI...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...
2,859112,https://cdns.overmind.ai/autorizacao-sulameric...,203941226,{'procedimentos_autorizados': ['SEPTOPLASTIA T...,\n Procedimento Autorizado\n Pacient...,"[{""code"":30501350,""description"":""RINOSSEPTOPLA...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...


In [55]:
llm_results_df.loc[0, 'AGENT RESULTS']

{'procedimentos_autorizados': ['VARIZES TRATAMENTO CIRURGICO DE DOIS MEMBROS'],
 'paciente': 'BARBARA ARAUJO SCHNNEPPEL CORREA',
 'codigo_autorizado': '30907136 x 1',
 'senha': 'J5VEYT7',
 'validade_senha': '07-01-2026',
 'data_solicitacao': '11-07-2025',
 'observacoes_opme': 'Pedido sem OPME',
 'observacoes_gerais': None,
 'profissional_solicitante': 'MATEUS ALVES BORGES CRISTINO'}

Part 4: get the llm results back to excel tab autorizacoes

In [ ]:
def merge_llm_results_to_excel(
    xls: pd.ExcelFile,
    llm_results_df: pd.DataFrame,
    output_excel_path: str,
    target_tabs: List[str],
) -> None:
    """
    Merge LLM results back to specific Excel tabs and save to a new file.
    
    Args:
        xls (pd.ExcelFile): Original Excel file object
        llm_results_df (pd.DataFrame): DataFrame containing LLM results with CD_AVISO_CIRURGIA as key
        output_excel_path (str): Path where the updated Excel file should be saved
        target_tabs (list, optional): List of tab names to merge LLM results with. 
                                    If None, merges with ["Autorizados", "Pendentes"]
    
    Returns:
        None: Saves the updated Excel file to the specified path
    """
    if target_tabs is None:
        raise ValueError("target_tabs must be provided as a list of tab names")
    
    # Dictionary to store processed dataframes
    processed_tabs = {}
    
    # Process each target tab
    for tab_name in target_tabs:
        if tab_name in xls.sheet_names:
            # Read the tab
            df_tab = pd.read_excel(xls, sheet_name=tab_name)
            
            # Merge with LLM results
            df_tab_final = df_tab.merge(llm_results_df, on="CD_AVISO_CIRURGIA", how="left")
            
            # Store the processed dataframe
            processed_tabs[tab_name] = df_tab_final
            
            logger.info(f"📊 Processed {tab_name}: {len(df_tab_final)} rows")
        else:
            logger.info(f"⚠️  Warning: Tab '{tab_name}' not found in Excel file")
    
    # Write all tabs back to Excel
    with pd.ExcelWriter(output_excel_path, engine="openpyxl", mode="w") as writer:
        for sheet in xls.sheet_names:
            sheet_str = str(sheet)
            if sheet in processed_tabs:
                # Write the processed tab with LLM results
                processed_tabs[sheet].to_excel(writer, sheet_name=sheet_str, index=False)
            else:
                # Write the original tab unchanged
                df_other = pd.read_excel(xls, sheet_name=sheet)
                df_other.to_excel(writer, sheet_name=sheet_str, index=False)
    
    logger.info(f"✅ Successfully saved updated Excel file to: {output_excel_path}")



In [57]:
# Use the function to merge results for both Autorizados and Pendentes tabs
from datetime import datetime
time_stamp = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
merge_llm_results_to_excel(
    xls=xls,
    llm_results_df=llm_results_df,
    output_excel_path=f"docs/excel/llm_output_enriquecido_{time_stamp}.xlsx",
    target_tabs=["Autorizados", "Pendentes", "Solicitados"]
)

📊 Processed Autorizados: 329 rows
📊 Processed Pendentes: 97 rows
📊 Processed Solicitados: 229 rows
✅ Successfully saved updated Excel file to: docs/excel/llm_output_enriquecido_01-09-2025 11:52:47.xlsx
✅ Successfully saved updated Excel file to: docs/excel/llm_output_enriquecido_01-09-2025 11:52:47.xlsx


upload back to s3